In [3]:
!pip install langchain chromadb openai tiktoken pypdf langchain_openai langchain-community langchain_huggingface

In [6]:
from langchain_community.vectorstores import Chroma

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_763/3884418650.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import Chroma


In [8]:
from langchain_core.documents import Document

# Create LangChain documents for IPL players

doc1 = Document(
        page_content="Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.",
        metadata={"team": "Royal Challengers Bangalore"}
    )
doc2 = Document(
        page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.",
        metadata={"team": "Mumbai Indians"}
    )
doc3 = Document(
        page_content="MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.",
        metadata={"team": "Chennai Super Kings"}
    )
doc4 = Document(
        page_content="Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.",
        metadata={"team": "Mumbai Indians"}
    )
doc5 = Document(
        page_content="Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.",
        metadata={"team": "Chennai Super Kings"}
    )

In [9]:
docs = [doc1, doc2, doc3, doc4, doc5]

In [12]:
from langchain_huggingface import HuggingFaceEmbeddings

vector_store = Chroma(
    embedding_function= HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
),
    persist_directory='my_chroma_db',
    collection_name='sample'
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

/tmp/ipykernel_763/3747884658.py:3: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vector_store = Chroma(


In [13]:
# add documents
vector_store.add_documents(docs)

['83efe448-48b6-4217-92b4-45c7694d70aa',
 '50de2710-14b4-4132-8d32-3796071bcd76',
 '64e4411d-1a36-4530-a3f8-bc316b081b53',
 '4449730a-09eb-4dd1-947b-5c876b2c273d',
 '23b8b4cf-00d4-4b6e-8ff0-8bb863d2ca10']

In [14]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['83efe448-48b6-4217-92b4-45c7694d70aa',
  '50de2710-14b4-4132-8d32-3796071bcd76',
  '64e4411d-1a36-4530-a3f8-bc316b081b53',
  '4449730a-09eb-4dd1-947b-5c876b2c273d',
  '23b8b4cf-00d4-4b6e-8ff0-8bb863d2ca10'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [15]:
# search documents
vector_store.similarity_search(
    query='Who among these are a bowler?',
    k=2
)

[Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
 Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure.")]

In [16]:
# search with similarity score
vector_store.similarity_search_with_score(
    query='Who among these are a bowler?',
    k=2
)

[(Document(metadata={'team': 'Mumbai Indians'}, page_content='Jasprit Bumrah is considered one of the best fast bowlers in T20 cricket. Playing for Mumbai Indians, he is known for his yorkers and death-over expertise.'),
  0.9693599343299866),
 (Document(metadata={'team': 'Mumbai Indians'}, page_content="Rohit Sharma is the most successful captain in IPL history, leading Mumbai Indians to five titles. He's known for his calm demeanor and ability to play big innings under pressure."),
  1.149344801902771)]

In [17]:
# meta-data filtering
vector_store.similarity_search_with_score(
    query="",
    filter={"team": "Chennai Super Kings"}
)

[(Document(metadata={'team': 'Chennai Super Kings'}, page_content='MS Dhoni, famously known as Captain Cool, has led Chennai Super Kings to multiple IPL titles. His finishing skills, wicketkeeping, and leadership are legendary.'),
  1.8436005115509033),
 (Document(metadata={'team': 'Chennai Super Kings'}, page_content='Ravindra Jadeja is a dynamic all-rounder who contributes with both bat and ball. Representing Chennai Super Kings, his quick fielding and match-winning performances make him a key player.'),
  1.890937328338623)]

In [18]:
# update documents
updated_doc1 = Document(
    page_content="Virat Kohli, the former captain of Royal Challengers Bangalore (RCB), is renowned for his aggressive leadership and consistent batting performances. He holds the record for the most runs in IPL history, including multiple centuries in a single season. Despite RCB not winning an IPL title under his captaincy, Kohli's passion and fitness set a benchmark for the league. His ability to chase targets and anchor innings has made him one of the most dependable players in T20 cricket.",
    metadata={"team": "Royal Challengers Bangalore"}
)

vector_store.update_document(document_id='09a39dc6-3ba6-4ea7-927e-fdda591da5e4', document=updated_doc1)


In [19]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['83efe448-48b6-4217-92b4-45c7694d70aa',
  '50de2710-14b4-4132-8d32-3796071bcd76',
  '64e4411d-1a36-4530-a3f8-bc316b081b53',
  '4449730a-09eb-4dd1-947b-5c876b2c273d',
  '23b8b4cf-00d4-4b6e-8ff0-8bb863d2ca10'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca

In [20]:
# delete document
vector_store.delete(ids=['09a39dc6-3ba6-4ea7-927e-fdda591da5e4'])

In [21]:
# view documents
vector_store.get(include=['embeddings','documents', 'metadatas'])

{'ids': ['83efe448-48b6-4217-92b4-45c7694d70aa',
  '50de2710-14b4-4132-8d32-3796071bcd76',
  '64e4411d-1a36-4530-a3f8-bc316b081b53',
  '4449730a-09eb-4dd1-947b-5c876b2c273d',
  '23b8b4cf-00d4-4b6e-8ff0-8bb863d2ca10'],
 'embeddings': array([[ 0.00994728,  0.06914336, -0.05147117, ..., -0.03543339,
          0.01284808,  0.01248293],
        [ 0.00127746,  0.03129853, -0.02375378, ..., -0.0051836 ,
         -0.03280611,  0.02737715],
        [-0.10265916,  0.02650813,  0.02271503, ..., -0.03359744,
         -0.07984944, -0.01507706],
        [ 0.02123395, -0.02468549, -0.04494376, ..., -0.10995813,
          0.00572561,  0.09915381],
        [ 0.0187398 ,  0.04382842, -0.04304253, ..., -0.0780162 ,
         -0.07840683, -0.00304189]]),
 'documents': ['Virat Kohli is one of the most successful and consistent batsmen in IPL history. Known for his aggressive batting style and fitness, he has led the Royal Challengers Bangalore in multiple seasons.',
  "Rohit Sharma is the most successful ca